<a href="https://colab.research.google.com/github/Rohim110606/BigData26_A_2411533015_Abdur-Rohim/blob/main/Praktikum2/BD_A_P02_2411533015_ABDUR_ROHIM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 17.8 MB/s eta 0:00:00


K1 Import Library dan Insialisai

In [ ]:
import numpy as np
import pandas as pd
from faker import Faker
import random

Kodingan diatas digunakan untuk mengimpor library NumPy dan Pandas yang berfungsi untuk mengolah data, serta Faker dan Random untuk membuat data dan nilai secara acak

K2 Membuat Dataset Sintetis

In [ ]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + " "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None]) # rating opsional
    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))


Jumlah baris: 515


Kodingan diatas berfungsi untuk membuat dataset transaksi sebanyak 500 data secara acak menggunakan Faker, NumPy, dan Random, yang berisi informasi pelanggan, produk, harga, jumlah, metode pembayaran, tanggal, kota, dan rating. Kode juga menambahkan missing value dan 15 data duplikat, kemudian mengacak data dan menyimpannya sebagai file transaksi_mentah.csv

K3 Deteksi dan Penanganan Value

In [ ]:
print (df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


Kodingan diatas berfungsi untuk menghitung jumlah nilai kosong pada setiap kolom dalam DataFrame df. Hasilnya digunakan untuk mengetahui kolom mana yang memiliki data yang belum terisi.

In [ ]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

K4 Deteksi dan Penanganan Duplikat

In [ ]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


Kodingan di atas berfungsi untuk menghitung jumlah data duplikat pada seluruh kolom dan transaction_id sebelum dilakukan pembersihan. Setelah itu, drop_duplicates() digunakan untuk menghapus data duplikat dan menampilkan jumlah baris yang tersisa.

K5 Koreksi Tipe Data dan Standarasi format

a. Standarisasi Teks Kategorikal

In [ ]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

Kodingan diatas berfungsi untuk membersihkan data teks pada kolom category, payment_method, dan shipping_city dengan menghapus spasi dan menyeragamkan huruf. Kemudian, kode mengubah penulisan “Cod” menjadi “COD” agar format metode pembayaran lebih konsisten.

b.Koreksi Tipe Data pada kolom price

In [ ]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

Kodingan diatas berfungsi untuk membersihkan dan mengubah format harga menjadi angka (float) dengan menghapus Rp, titik, koma, dan spasi yang tidak diperlukan. Jika terdapat data kosong atau format harga yang tidak valid, kode akan mengubahnya menjadi NaN

c. Standarisasi format tanggal

In [ ]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

Kodingan diatas berfungsi untuk mengubah berbagai format tanggal menjadi format yang seragam, yaitu YYYY-MM-DD. Jika format tanggal tidak sesuai, data akan dianggap tidak valid (NaT)

K6 Ekspor Dataset Bersih

In [ ]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


Kodingan diatas berfungsi untuk menyimpan dataset yang sudah dibersihkan ke dalam file transaksi_bersih.csv. Kemudian, kode menampilkan jumlah baris data yang tersimpan dalam dataset tersebut.

In [ ]:
# Upload file CSV ke Google Drive

from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Buat folder khusus (jika belum ada)
folder_path = "/content/drive/MyDrive/BigData/Praktikum2"
os.makedirs(folder_path, exist_ok=True)

# Copy file ke folder tersebut
!cp transaksi_mentah.csv "{folder_path}/"
!cp transaksi_bersih.csv "{folder_path}/"

print("✅ File berhasil diupload ke Google Drive!")
print(f"📁 Lokasi: MyDrive/BigData/Praktikum2")
print(f"📄 File yang diupload:")
print(f"   - transaksi_mentah.csv")
print(f"   - transaksi_bersih.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ File berhasil diupload ke Google Drive!
📁 Lokasi: MyDrive/BigData/Praktikum2
📄 File yang diupload:
   - transaksi_mentah.csv
   - transaksi_bersih.csv
